# 03 — Claude Code, MCP & Integration

Code-first experiments for the module’s configuration decisions. Start with the [22-screen module index](../course%20content%20HTML/03-claude-code-mcp-integration/index.html#navigation-sections).


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'study_support.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from study_support import load_anthropic_api_key, messages_create
assert callable(load_anthropic_api_key)
assert callable(messages_create)


## Session, headless, commands, and memory

Course source: [skills and custom commands](../course%20content%20HTML/03-claude-code-mcp-integration/04-packaging-workflows.html#s08) · [course S08](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/168108zug1cy3/Developer_M3_vF2.html#S08). Exam-guide supplement: [Claude Code operation](../course%20content%20HTML/exam-guide.html#3-claude-code). Keep durable project rules in files, reusable invocations in commands/Skills, and run non-interactive automation headlessly with explicit session behavior.


In [ ]:
def claude_invocation(prompt, *, headless=False, resume=None):
    command = ["claude"]
    if headless:
        command += ["-p", prompt, "--output-format", "json"]
    else:
        command += [prompt]
    if resume:
        command += ["--resume", resume]
    return command

memory_scope = {"CLAUDE.md": "project", "rules/security.md": "path-scoped", "session": "ephemeral"}
assert claude_invocation("review", headless=True) == ["claude", "-p", "review", "--output-format", "json"]
assert claude_invocation("continue", resume="session-123")[-2:] == ["--resume", "session-123"]
assert memory_scope["session"] == "ephemeral"


## Permission modes and human gates

Source: [agent loop and permission modes](../course%20content%20HTML/03-claude-code-mcp-integration/02-permission-modes-human-gates.html#s02) · [course S02](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/168108zug1cy3/Developer_M3_vF2.html#S02) and [Checkpoint 1](../course%20content%20HTML/03-claude-code-mcp-integration/02-permission-modes-human-gates.html#s04). Change the action or reversibility and rerun.


In [ ]:
READ_ONLY = {"inspect", "search", "plan"}

def permission_decision(action, *, reversible=True):
    if action in READ_ONLY:
        return "allow"
    return "allow" if reversible else "ask"

def review_gate(cost_of_error):
    return "human" if cost_of_error == "high" else "automated"

assert permission_decision("inspect", reversible=False) == "allow"
assert permission_decision("deploy", reversible=False) == "ask"
assert review_gate("high") == "human"


## Minimal MCP JSON-RPC over stdio

Source: [MCP servers and stdio transport](../course%20content%20HTML/03-claude-code-mcp-integration/05-mcp-servers.html#s12) · [course S12](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/168108zug1cy3/Developer_M3_vF2.html#S12). This launches a tiny server process, sends a JSON-RPC `tools/list` request on stdin, and validates the response.


In [ ]:
import json
import subprocess
import sys

server = r'''
import json, sys
request = json.loads(sys.stdin.readline())
assert request["jsonrpc"] == "2.0" and request["method"] == "tools/list"
response = {"jsonrpc": "2.0", "id": request["id"], "result": {"tools": [{"name": "lookup_order", "description": "Read one order by ID", "inputSchema": {"type": "object", "properties": {"order_id": {"type": "integer"}}, "required": ["order_id"]}}]}}
print(json.dumps(response))
'''
request = {"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}
run = subprocess.run([sys.executable, "-c", server], input=json.dumps(request) + "\n", text=True, capture_output=True, check=True)
response = json.loads(run.stdout)
assert response["id"] == 1
assert response["result"]["tools"][0]["name"] == "lookup_order"


## Durable project context

Source: [CLAUDE.md, rules, hooks, and subagents](../course%20content%20HTML/03-claude-code-mcp-integration/03-durable-project-context.html#s05) · [course S05](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/168108zug1cy3/Developer_M3_vF2.html#S05) and [Checkpoint 2](../course%20content%20HTML/03-claude-code-mcp-integration/03-durable-project-context.html#s07). Merge broad-to-specific settings, then prove the hook blocks one destructive command.


In [ ]:
import shlex

def merge_config(*broad_to_specific):
    merged = {}
    for layer in broad_to_specific:
        merged.update(layer)
    return merged

BLOCKED_PREFIXES = [("git", "reset", "--hard"), ("git", "clean", "-fd")]

def hook_allows(command):
    tokens = tuple(shlex.split(command))
    return not any(tokens[:len(prefix)] == prefix for prefix in BLOCKED_PREFIXES)

effective = merge_config(
    {"style": "concise", "tests": False},
    {"tests": True, "language": "python"},
    {"security": "strict"},
)
assert effective["tests"] is True
assert hook_allows("git status")
assert not hook_allows("git reset --hard HEAD~1")


## Portable workflow packaging

Source: [skills, commands, plugins, and runtime loading](../course%20content%20HTML/03-claude-code-mcp-integration/04-packaging-workflows.html#s08) · [course S08](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/168108zug1cy3/Developer_M3_vF2.html#S08) and [the broken absolute-path checkpoint](../course%20content%20HTML/03-claude-code-mcp-integration/04-packaging-workflows.html#s11). Try an absolute or parent-traversing path and confirm it is rejected.


In [ ]:
from pathlib import PurePosixPath

def portable_project_path(path):
    candidate = PurePosixPath(path)
    if candidate.is_absolute() or ".." in candidate.parts:
        raise ValueError("skill paths must stay relative to the project root")
    return f"$CLAUDE_PROJECT_DIR/{candidate}"

assert portable_project_path("scripts/validate-migration.sh") == "$CLAUDE_PROJECT_DIR/scripts/validate-migration.sh"
try:
    portable_project_path("/Users/alex/scripts/validate.sh")
except ValueError:
    pass
else:
    raise AssertionError("absolute paths must be rejected")


## MCP capability, transport, and scope

Source: [MCP servers, transport, scope, and tool permissions](../course%20content%20HTML/03-claude-code-mcp-integration/05-mcp-servers.html#s12) · [course S12](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/168108zug1cy3/Developer_M3_vF2.html#S12) and [Checkpoint 5](../course%20content%20HTML/03-claude-code-mcp-integration/05-mcp-servers.html#s14). Change the deployment facts and keep the result least-privileged.


In [ ]:
def capability_for(intent):
    return {"act": "tool", "read": "resource", "template": "prompt"}[intent]

def connection_decision(*, same_machine, shared, personal_credentials):
    transport = "stdio" if same_machine else "HTTP"
    scope = "local" if personal_credentials or not shared else "project"
    return transport, scope

assert capability_for("read") == "resource"
assert capability_for("act") == "tool"
assert connection_decision(same_machine=True, shared=False, personal_credentials=True) == ("stdio", "local")
assert connection_decision(same_machine=False, shared=True, personal_credentials=False) == ("HTTP", "project")


## Hands-on exercise: security-review an enterprise connection

Source: [enterprise authentication and modernization](../course%20content%20HTML/03-claude-code-mcp-integration/06-enterprise-integration.html#s15) · [course S15](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/168108zug1cy3/Developer_M3_vF2.html#S15) and the [cumulative bug-identification task](../course%20content%20HTML/03-claude-code-mcp-integration/07-cumulative-integration-task.html#s18). Add one unsafe choice to `secure`, rerun, and explain the new gap before restoring it.


In [ ]:
def integration_gaps(spec):
    checks = {
        "permission mode": spec["permission_mode"] != "bypassPermissions",
        "portable skill path": not spec["skill_path"].startswith("/"),
        "secret separation": spec["credential"].startswith("$"),
        "human production gate": spec["production_gate"] == "human",
    }
    return [name for name, passed in checks.items() if not passed]

secure = {
    "permission_mode": "acceptEdits",
    "skill_path": "$CLAUDE_PROJECT_DIR/scripts/validate-migration.sh",
    "credential": "$WAREHOUSE_MCP_TOKEN",
    "production_gate": "human",
}
assert integration_gaps(secure) == []


## Hands-on exercise: choose the smallest reusable tool surface

Source: [Skills, commands, and plugins](../course%20content%20HTML/03-claude-code-mcp-integration/04-packaging-workflows.html#s08) · [course S08](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/168108zug1cy3/Developer_M3_vF2.html#S08) and [MCP server boundary](../course%20content%20HTML/03-claude-code-mcp-integration/05-mcp-servers.html#s12) · [course S12](https://anthropic-partners.skilljar.com/content/wp/4hdejjwplbrm/168108zug1cy3/Developer_M3_vF2.html#S12). Add a scenario and assertion: built-in for native capability, custom tool for one app-owned action, Skill for a reusable workflow, MCP for an independently maintained capability shared across applications.


In [ ]:
def tool_surface(*, native=False, workflow=False, shared_service=False):
    if native:
        return "built-in"
    if shared_service:
        return "MCP"
    if workflow:
        return "Skill"
    return "custom tool"

assert tool_surface(native=True) == "built-in"
assert tool_surface(shared_service=True) == "MCP"
assert tool_surface(workflow=True) == "Skill"
assert tool_surface() == "custom tool"
